# Lab 10.1: ChatGPT-Scale System Design Calculator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.1_chatgpt_scale_chatbot/lab.ipynb)
[![Open In Molab](https://raw.githubusercontent.com/marimo-team/marimo/main/docs/_static/marimo-badge.svg)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.1_chatgpt_scale_chatbot/lab.ipynb)

Calculate GPU fleet size, memory budgets, caching impact, and cost per conversation for a ChatGPT-scale system.

In [ ]:
# Install dependencies
import subprocess, sys
# Install numpy and matplotlib for fleet sizing calculations and charts
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'matplotlib'])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === SYSTEM PARAMETERS (change these and re-run) ===
# Number of users with active sessions simultaneously
CONCURRENT_USERS = 1_000_000
# Average seconds between user messages (think time)
THINK_TIME_S = 45
# Average tokens generated per assistant response
TOKENS_PER_RESPONSE = 256
# Average context window accumulated per conversation
CONTEXT_LENGTH = 4096
# Fraction of traffic routed to large model (complex queries)
QUALITY_FRACTION = 0.20
# GPU memory in bytes
GPU_HBM_GB = 80

In [ ]:
def compute_fleet_size():
    """Calculate minimum GPU fleet from first principles."""
    
    # Step 1: Derive request rate from concurrent users and think time
    requests_per_sec = CONCURRENT_USERS / THINK_TIME_S  # ~22K req/s
    # Total tokens per second the system must generate
    tokens_per_sec = requests_per_sec * TOKENS_PER_RESPONSE  # ~5.7M tok/s
    
    # Step 2: Model memory configurations
    # 70B INT4 model (quality tier)
    weights_70b_gb = 35  # 70B params * 0.5 bytes (INT4)
    # KV cache per token for 70B with GQA (8 KV heads, 128 dim, 80 layers, FP16)
    kv_per_token_70b = 2 * 8 * 128 * 80 * 2  # 327,680 bytes = 0.31 MB/token
    # KV cache per user at full context length
    kv_per_user_70b_gb = CONTEXT_LENGTH * kv_per_token_70b / 1e9  # ~1.28 GB
    
    # 8B INT8 model (cost tier)
    weights_8b_gb = 8  # 8B params * 1 byte (INT8)
    # KV per token for 8B with GQA (8 KV heads, 128 dim, 32 layers, FP16)
    kv_per_token_8b = 2 * 8 * 128 * 32 * 2  # 131,072 bytes = 0.125 MB/token
    # KV per user at full context
    kv_per_user_8b_gb = CONTEXT_LENGTH * kv_per_token_8b / 1e9  # ~0.5 GB
    
    # Step 3: Users per GPU (memory-limited)
    # Available KV memory = total HBM - weights - overhead
    avail_kv_70b = GPU_HBM_GB - weights_70b_gb - 7  # 38 GB for KV
    avail_kv_8b = GPU_HBM_GB - weights_8b_gb - 10  # 62 GB for KV
    users_per_gpu_70b = int(avail_kv_70b / kv_per_user_70b_gb)  # ~29
    users_per_gpu_8b = int(avail_kv_8b / kv_per_user_8b_gb)  # ~124
    
    # Step 4: Naive fleet size (all users loaded simultaneously)
    quality_users = int(CONCURRENT_USERS * QUALITY_FRACTION)  # 200K
    cost_users = CONCURRENT_USERS - quality_users  # 800K
    naive_gpus_quality = int(np.ceil(quality_users / users_per_gpu_70b))
    naive_gpus_cost = int(np.ceil(cost_users / users_per_gpu_8b))
    naive_total = naive_gpus_quality + naive_gpus_cost
    
    # Step 5: Oversubscription (only ~7% actively generating at any instant)
    # Active fraction = generation_time / (generation_time + think_time)
    gen_time_s = TOKENS_PER_RESPONSE * 0.05  # ~50ms ITL * 256 tokens = 12.8s
    active_fraction = gen_time_s / (gen_time_s + THINK_TIME_S)  # ~22%
    # With KV swap to CPU for idle users, oversubscribe 3x safely
    oversubscription = 3.0  # idle users' KV parked in CPU memory
    optimized_total = int(np.ceil(naive_total / oversubscription))
    
    # Print the sizing breakdown
    print('=== Fleet Sizing ===')
    print(f'Request rate: {requests_per_sec:,.0f} req/s')
    print(f'Token rate: {tokens_per_sec/1e6:.1f}M tokens/s')
    print(f'\n70B tier: {kv_per_user_70b_gb:.2f} GB/user, {users_per_gpu_70b} users/GPU')
    print(f'8B tier: {kv_per_user_8b_gb:.2f} GB/user, {users_per_gpu_8b} users/GPU')
    print(f'\nNaive fleet: {naive_total:,} GPUs (all users loaded)')
    print(f'Active fraction: {active_fraction:.1%}')
    print(f'Optimized fleet: {optimized_total:,} GPUs (with {oversubscription:.0f}x oversubscription)')
    
    return {
        'naive_total': naive_total,
        'optimized_total': optimized_total,
        'users_per_gpu_70b': users_per_gpu_70b,
        'users_per_gpu_8b': users_per_gpu_8b,
        'kv_per_user_70b_gb': kv_per_user_70b_gb,
        'kv_per_user_8b_gb': kv_per_user_8b_gb,
        'requests_per_sec': requests_per_sec,
        'tokens_per_sec': tokens_per_sec,
    }

# Run the fleet sizing calculation
sizing = compute_fleet_size()

In [ ]:
def plot_memory_budget():
    """Visualize GPU memory allocation for both model tiers."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # 70B INT4 memory breakdown
    labels_70b = ['Weights\n(INT4)', 'KV Cache\nPool', 'Activations', 'CUDA\nOverhead']
    sizes_70b = [35, 38, 2, 5]  # GB allocations for 70B model on 80GB GPU  # GB allocations for 70B model
    colors_70b = ['#f3e8ff', '#dbeafe', '#fef3c7', '#f3f4f6']  # purple, blue, amber, gray
    # Explode the KV cache slice to highlight it (largest usable portion)
    # Explode KV cache slice to highlight it (largest usable portion)
    explode_70b = (0, 0.05, 0, 0)
    
    axes[0].pie(sizes_70b, labels=labels_70b, colors=colors_70b, explode=explode_70b,
               autopct='%1.0f%%', startangle=90, textprops={'fontsize': 10})
    axes[0].set_title(f'70B INT4 (A100 80GB)\n{sizing["users_per_gpu_70b"]} users/GPU')
    
    # 8B INT8 memory breakdown
    labels_8b = ['Weights\n(INT8)', 'KV Cache\nPool', 'Activations', 'Overhead +\nPrefix Cache']
    sizes_8b = [8, 62, 1.5, 8.5]  # GB allocations for 8B model on 80GB GPU  # GB allocations for 8B model
    colors_8b = ['#f3e8ff', '#dbeafe', '#fef3c7', '#f3f4f6']
    explode_8b = (0, 0.05, 0, 0)
    
    axes[1].pie(sizes_8b, labels=labels_8b, colors=colors_8b, explode=explode_8b,
               autopct='%1.0f%%', startangle=90, textprops={'fontsize': 10})
    axes[1].set_title(f'8B INT8 (A100 80GB)\n{sizing["users_per_gpu_8b"]} users/GPU')
    
    plt.suptitle('GPU Memory Budget: Two-Tier Architecture', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

# Show memory allocation pie charts
plot_memory_budget()

In [ ]:
def plot_caching_impact():
    """Show how each caching layer reduces the GPU fleet requirement."""
    
    # Starting point: fleet without any caching optimization
    base_gpus = 1500  # baseline fleet before caching
    
    # Each cache layer eliminates a portion of GPU demand
    layers = [
        ('No caching\n(baseline)', base_gpus, '#ffe4e6'),     # starting point
        ('+ Prefix cache\n(-380 GPUs)', base_gpus - 380, '#fef3c7'),  # shared system prompt
        ('+ Session KV\n(-200 GPUs)', base_gpus - 580, '#dbeafe'),    # multi-turn persistence
        ('+ Semantic cache\n(-100 GPUs)', base_gpus - 680, '#dcfce7'), # identical query dedup
    ]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    names = [l[0] for l in layers]  # layer names for x-axis
    gpus = [l[1] for l in layers]   # GPU count at each stage
    colors = [l[2] for l in layers]  # color per bar
    
    bars = ax.bar(names, gpus, color=colors, edgecolor='black', linewidth=1.2)
    
    # Annotate bars with GPU count and percentage reduction
    for i, (bar, gpu) in enumerate(zip(bars, gpus)):
        # Show absolute GPU count on bar
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{gpu} GPUs', ha='center', fontsize=10, fontweight='bold')
        if i > 0:
            # Show percentage saved compared to baseline
            pct_saved = (1 - gpu/base_gpus) * 100
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
                    f'-{pct_saved:.0f}%', ha='center', fontsize=11, color='#991b1b')
    
    ax.set_ylabel('GPUs Required')  # total fleet size
    ax.set_title('Cumulative Impact of Three-Layer Caching')
    ax.set_ylim(0, 1800)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    
    # Print the savings summary
    total_saved = base_gpus - gpus[-1]  # GPUs eliminated by all caching
    print(f'Total GPU reduction: {total_saved} GPUs ({total_saved/base_gpus:.0%} savings)')

# Visualize caching impact
plot_caching_impact()

In [ ]:
def plot_cost_per_conversation():
    """Calculate and visualize cost per conversation across optimization stages."""
    
    # Cost model parameters
    h100_rate = 3.50  # $/hr per H100
    a100_rate = 2.00  # $/hr per A100
    
    # Conversations per day: requests/sec * seconds/day / turns_per_conversation
    convos_per_day = sizing['requests_per_sec'] * 86400 / 10  # 10 turns per convo
    
    # Four optimization stages with their fleet configurations
    stages = [
        # (label, h100s, a100s, instance_discount)
        ('Naive\n(no optimization)', 6000, 6000, 1.0),   # all users loaded, no sharing
        ('+ Tiering\n(8B for 80%)', 1200, 600, 1.0),      # two-tier routing
        ('+ Caching\n(3 layers)', 500, 600, 1.0),         # prefix+session+semantic
        ('+ Instance mix\n(reserved+spot)', 500, 600, 0.6), # 40% discount from mix
    ]
    
    # Calculate hourly cost and cost/conversation for each stage
    hourly_costs = []  # total fleet cost per hour
    cost_per_convo = []  # cost per single conversation
    for label, h100s, a100s, discount in stages:
        hourly = (h100s * h100_rate + a100s * a100_rate) * discount
        # Cost per conversation = hourly_cost / (conversations_per_hour)
        cpc = hourly / (convos_per_day / 24)  # cost per conversation
        hourly_costs.append(hourly)
        cost_per_convo.append(cpc)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    labels = [s[0] for s in stages]  # stage names
    # Use log scale because costs span 3 orders of magnitude
    colors = ['#ffe4e6', '#fef3c7', '#dbeafe', '#dcfce7']
    
    bars = ax.bar(labels, [c * 1000 for c in cost_per_convo],  # convert to milli-dollars
                  color=colors, edgecolor='black', linewidth=1.2)
    
    # Annotate with actual dollar values
    for bar, cpc in zip(bars, cost_per_convo):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'${cpc:.5f}', ha='center', fontsize=9)
    
    # Add target line at $0.005/conversation
    ax.axhline(y=5.0, color='red', linestyle='--', linewidth=1.5, label='$0.005 budget')
    
    ax.set_ylabel('Cost per Conversation (milli-$)')  # thousandths of a dollar
    ax.set_title('Cost Optimization: From Naive to Production')
    ax.legend()
    ax.set_yscale('log')  # log scale to show all stages clearly
    plt.tight_layout()
    plt.show()
    
    # Print final cost summary
    print(f'Final cost: ${cost_per_convo[-1]:.6f}/conversation')
    print(f'Monthly cost: ${hourly_costs[-1] * 730:,.0f}')
    print(f'Conversations/month: {convos_per_day * 30 / 1e9:.2f}B')

# Visualize cost optimization stages
plot_cost_per_conversation()